# CorroborIA — exploration, approche et résultats

Ce notebook rejoue la corroboration **Système A (RH) ↔ Système B (Temps)** sur les extractions fournies
et illustre les trois types de cas exigés : **conforme**, **écart justifié automatiquement**, **vraie anomalie**.

Approche en 3 niveaux : comparaison brute normalisée → règles métier du mapping → analyse IA des cas ambigus
(patterns + scikit-learn + LLM avec repli), puis boucle de rétroaction expert.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from corroborai.io_loader import load
ds = load()
print({k: len(v) for k, v in {"source": ds.source, "destination": ds.destination, "detail": ds.detail, "motif": ds.motif}.items()})
print("Intégrité :", all(i["intact"] for i in ds.integrity))

{'source': 23, 'destination': 22, 'detail': 114, 'motif': 92}
Intégrité : True


## 1. Exploration
- La source a **23 affectations** (un employé peut en avoir plusieurs : P primaire, A temporaire, S secondaire) ; la cible **22**.
- Le fichier *détail du poste* est un CSV tassé dans une seule colonne Excel → reparsé.
- Les dates sources sont des **sériels Excel**, `onboardDate` cible est en **ISO 8601**.
- La cible contient un défaut d'encodage (`Absence complÃ¨te`).

In [2]:
ds.mapping[["description", "champ_a", "champ_b", "ligne_excel"]]

,description,champ_a,champ_b,ligne_excel
0,Prénom de l’employé,PrénomUsuel,givenName,2
1,Nom de l’employé,NomFamille,surname,3
2,Email de l’employé,Adresse courriel travail,contactEmail,4
3,Date d’embauche,DateEmbaucheRécente,onboardDate,7
4,Code de l’employé,Matricule,personId,8
5,Emplacement,LibelléSite,siteName,9
6,Code d’emplacement,CodeSite,siteCode,10
7,Département,CodeDirection,divisionId,11
8,Département,LibelléDirection,divisionName,12
9,Code de département,CodeImputation,divisionCode,13


In [3]:
ds.destination["detailedStatus"].value_counts()

detailedStatus
Actif                20
Absence complÃ¨te     2
Name: count, dtype: int64

## 2. Exécution du moteur (sans LLM externe : IA locale uniquement, reproductible)

In [4]:
from corroborai.engine import Engine
from corroborai.ai.feedback import FeedbackStore
import tempfile
res = Engine(use_llm=False, feedback=FeedbackStore(pathlib.Path(tempfile.mkdtemp()) / "fb.json")).run()
res.counts()

{'total': 551,
 'Conforme': 361,
 'Écart justifié': 152,
 'Anomalie': 38,
 'a_valider': 28}

In [5]:
from corroborai.report import to_frame
df = to_frame(res.findings)
pd.crosstab(df["Champ Système B"], df["Verdict"])

Verdict,Anomalie,Conforme,Écart justifié
Champ Système B,,,
(affectation),1,0,0
assignmentEndDate,0,22,0
assignmentStartDate,3,19,0
contactEmail,22,0,0
contractTypeCode,4,0,18
dailyHoursOverride,3,18,1
detailedStatus,0,0,22
divisionCode,0,22,0
divisionId,0,22,0


## 3. Les trois types de cas

In [6]:
def show(mat, champ):
    f = next(f for f in res.findings if f.matricule == mat and f.champ_b == champ)
    return pd.Series({"verdict": f.verdict, "niveau": f.niveau, "valeur A": f.valeur_a, "attendue": f.valeur_attendue,
                      "valeur B": f.valeur_b, "règle": f.regle_id, "justification": f.justification, "diagnostic": f.diagnostic})
show("8142123", "onboardDate")   # CONFORME : sériel Excel 31291 ≡ 1985-09-01T00:00:00.000Z

verdict                                                                                                          Conforme
niveau                                                                                              1 - Comparaison brute
valeur A                                                                                                       1985-09-01
attendue                                                                                                       1985-09-01
valeur B                                                                                         1985-09-01T00:00:00.000Z
règle                                                                                                            R-DIRECT
justification    Valeurs identiques après normalisation du format (date) : « 1985-09-01 » ≡ « 1985-09-01T00:00:00.000Z ».
diagnostic                                                                                                               
dtype: str

In [7]:
show("7603160", "statusReasonCode")   # ÉCART JUSTIFIÉ : jointure Motif 807 → code Remphor 170

verdict                                                                                                                   Écart justifié
niveau                                                                                                                  2 - Règle métier
valeur A                                                                                                                             807
attendue                                                                                                                             170
valeur B                                                                                                                             170
règle                                                                                                                       R-STATUS-CAD
justification    Écart expliqué par la règle R-STATUS-CAD : Absence (code 2) → jointure Motif : CodeRaisonStatut 807 → code Remphor 170.
diagnostic                               

In [8]:
show("2762457", "contractTypeCode")   # VRAIE ANOMALIE : V / permanent / temps plein → JWN attendu, WHX reçu

verdict                                                                                                                         Anomalie
niveau                                                                                                                  2 - Règle métier
valeur A                                                                        CatégorieEmploi=V | EstPermanent=Oui | EstTempsPlein=Oui
attendue                                                                                                                             JWN
valeur B                                                                                                                             WHX
règle                                                                                                                         R-CONTRACT
justification    Non-respect de la règle R-CONTRACT : valeur attendue « JWN », reçue « WHX ». SI PERM_IND=1 et FT_IND=1 et EMPTP_CD="...
diagnostic       Le code reçu WHX corresp

## 4. Cas ambigus traités par l'IA
- **positionName** : libellés pseudonymisés ; on vérifie que la correspondance code ↔ libellé est **biunivoque** sur tout le jeu.
- **Heures** : la cible conserve les heures du contrat du poste ; si la source est vide c'est un défaut légitime, sinon anomalie *à valider*.

In [9]:
amb = df[df["Niveau de décision"].str.startswith("3")]
amb.groupby(["Champ Système B", "Verdict", "Motif"]).size()

Champ Système B      Verdict         Motif                             
dailyHoursOverride   Anomalie        hours:override_non_transmis            3
                     Écart justifié  hours:defaut_poste_source_vide         1
positionName         Écart justifié  posname:pseudonymisation_coherente    22
weeklyHoursOverride  Anomalie        hours:override_non_transmis            3
                     Écart justifié  hours:defaut_poste_source_vide         1
dtype: int64

### Courriel : règle précisée par Loto-Québec
Le « code » de la règle est le **matricule**, et le préfixe d'environnement `dev-08-v2_` est **optionnel côté destination** :
il est accepté par la règle (niveau 2). Les 22 adresses respectent la structure mais utilisent un identifiant autre que le
matricule : anomalies **à valider**, priorité la plus basse (erreur de construction ou artefact d'anonymisation).

In [10]:
f = next(f for f in res.findings if f.champ_b == "contactEmail")
pd.Series({"verdict": f.verdict, "niveau": f.niveau, "à valider": f.a_valider, "priorité": f.priorite,
           "attendu": f.valeur_attendue, "reçu": f.valeur_b, "signaux": f.ia["signaux"]})

verdict                                                                                                                     Anomalie
niveau                                                                                                              2 - Règle métier
à valider                                                                                                                       True
priorité                                                                                                                          51
attendu                                                                                               pnom2911996996@loto-quebec.com
reçu                                                                                          dev-08-v2_PNom10446446@loto-quebec.com
signaux      [préfixe d'environnement optionnel « dev-08-v2_ » accepté (précision de Loto-Québec), structure « initiale + nom + i...
dtype: object

## 5. Calibration d'une règle ambiguë
Le mapping indique « date la plus **ancienne** » pour `assignmentStartDate`, mais les données du Système B suivent
massivement l'interprétation « plus **récente** ». Le moteur mesure les deux et documente le choix.

In [11]:
res.meta["calibration_regles"]

[{'regle': 'R-ASSIGN-START',
  'texte_mapping': "« Date la plus ancienne entre la date calculée du changement d'unité administrative et la date d'effet poste »",
  'interpretations': {'littérale (plus ancienne)': '0/22',
   'plus récente': '19/22'},
  'retenue': 'max',
  'conclusion': "Les données du Système B suivent massivement l'interprétation « plus récente » : elle est retenue (paramètre ASSIGN_START_MODE). À confirmer avec l'équipe fonctionnelle."}]

## 6. Anomalies priorisées (score ML)

In [12]:
df[df["Verdict"] == "Anomalie"][["Priorité", "Matricule", "Champ Système B", "Valeur attendue (règle)", "Valeur B (cible)", "Confiance", "À valider", "Diagnostic / cause probable"]]

,Priorité,Matricule,Champ Système B,Valeur attendue (règle),Valeur B (cible),Confiance,À valider,Diagnostic / cause probable
0,91,1545850,(affectation),1 enregistrement,NaN,0.99,,"L'employé existe dans le Système B (1 affectation(s) transmise(s)) mais son affectation temporaire (poste 45985, emp..."
1,83,2762457,contractTypeCode,JWN,WHX,0.97,,"Le code reçu WHX correspond à la condition « SI EMPTP_CD=""O"" (Occasionnel - Surnuméraire) --> Mettre ""WHX"" », alors ..."
2,83,3712987,contractTypeCode,JWN,XFLR,0.97,,"Le code reçu XFLR correspond à la condition « SI PERM_IND=1 et FT_IND=0 et EMPTP_CD=""V"" --> Mettre ""XFLR"" », alors q..."
3,83,4625374,contractTypeCode,WHX,JWN,0.97,,"Le code reçu JWN correspond à la condition « SI PERM_IND=1 et FT_IND=1 et EMPTP_CD=""V"" --> Mettre ""JWN"" », alors que..."
4,83,7254364,contractTypeCode,XFLR,JWN,0.97,,"Le code reçu JWN correspond à la condition « SI PERM_IND=1 et FT_IND=1 et EMPTP_CD=""V"" --> Mettre ""JWN"" », alors que..."
5,83,9989151,assignmentStartDate,2009-03-30,2021-03-30,0.97,,La cible reprend la date d'effet du DERNIER détail du poste 31109 (2021-03-30) (aucun changement d'unité adm. dans l...
6,82,4402456,assignmentStartDate,2002-06-10,2013-10-20,0.97,,La cible reprend la date d'effet du DERNIER détail du poste 26586 (2013-10-20) (aucun changement d'unité adm. dans l...
7,80,3241002,assignmentStartDate,2009-03-30,2022-10-05,0.97,,La cible reprend la date d'effet du DERNIER détail du poste 88622 (2022-10-05) (aucun changement d'unité adm. dans l...
8,71,3241002,siteName,Emplacement48,Emplacement35,0.97,,Le libellé reçu « Emplacement35 » correspond au site 35 alors que le code site est 48 (source et cible) : incohérenc...
9,70,2911996,dailyHoursOverride,7,8,0.60,Oui,Le Système B conserve les heures contractuelles du poste (8) au lieu de la norme de l'employé (7) : l'override ne se...


In [13]:
res.meta["synthese"]["resume"]

'38 anomalie(s) réelle(s) à investiguer sur 23 affectations ; 152 écart(s) justifié(s) automatiquement. Principales zones : (affectation) (1), contractTypeCode (4), assignmentStartDate (3), siteName (2).'

## 7. Export

In [14]:
from corroborai.report import export
export(res, pathlib.Path("../output"))

{'excel': WindowsPath('../output/rapport_corroboration.xlsx'),
 'csv': WindowsPath('../output/rapport_corroboration.csv'),
 'csv_anomalies': WindowsPath('../output/anomalies.csv')}